# Chapter 18 – Recursive versus iterative (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch18_recursive_vs_iterative_python.ipynb)

The algorithms of the chapter, recursive and iterative, compared with **counts** (calls, iterations, comparisons, recursion depth) and with some measured times: **SumExplicitStack** (recursion simulated with a stack), recursive **MergeSort** and **MergeSortBottomUp**, the recursive fast power and **PowerIt**, **Kadane**, the three Fibonacci algorithms **FibRec**, **FibMemo** and **FibIt**, **GCDRec**/**GCDIt** and the recursive and iterative binary search (tail-recursion elimination). The counts are the same in every language and on every machine; the times (in ms) depend on the machine and on the language, only how they grow matters.

The full chapter, with pseudo-code, proofs and worked examples, is [on the website](https://fabiofurini.github.io/algorithms/recursion/18-recursive-vs-iterative/).

Run each code cell with **Shift+Enter**, in order from top to bottom. Python 3, no external libraries.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `FibRec` → `fib_rec` (snake_case).

## 1. From recursion to an explicit stack: SumExplicitStack

**SumExplicitStack**$(a,n)$ — *Input*: an array $a[1..n]$ of $n\ge1$ integers. *Output*: the sum $a[1]+\dots+a[n]$. The iterative version of the halving sum: the stack $P$ contains the pairs $(l,r)$ of the sub-arrays still to be summed and does the job of the call stack. The stack never contains more than $\Theta(\log_2 n)$ pairs.

On the array $a=[4,7,1,9]$ of the chapter we print the trace (with indices from 0: the pair $(1,4)$ of the chapter is here $(0,3)$; in each row the top of the stack is on the right) and we compare with `SumIterative`, `sumRecHalving` and `sumRecDecrease` of the code of the course, with the maximum depth of the recursion. Then we print the call stack of `sumRecDecrease(a, 3)` on $a=[4,7,1]$ (example of the chapter): the additions are performed only *after* the base case, while the stack is emptied, and at some point the active calls are $n+1$.

In [1]:
max_p = 0   # maximum size of the stack P

def sum_explicit_stack(a, n):
    """SumExplicitStack(a, n): sum of a[0..n-1] with an explicit stack."""
    global max_p
    s = 0
    P = []                          # stack of the sub-arrays to be summed
    P.append((0, n - 1))
    max_p = max(max_p, len(P))
    while len(P) > 0:
        l, r = P.pop()
        if l == r:
            s = s + a[l]            # base case
        else:
            mid = l + (r - l) // 2
            P.append((mid + 1, r))
            P.append((l, mid))
            max_p = max(max_p, len(P))
    return s

def fmt_p(P):
    return "[" + ",".join(f"({x},{y})" for x, y in P) + "]"

def sum_explicit_stack_trace(a, n):
    s = 0
    P = [(0, n - 1)]
    while len(P) > 0:
        l, r = P.pop()
        if l == r:
            s = s + a[l]
            print(f"pop ({l},{r}): base case, s={s} => P={fmt_p(P)}")
        else:
            mid = l + (r - l) // 2
            P.append((mid + 1, r))
            P.append((l, mid))
            print(f"pop ({l},{r}): mid={mid}, push ({mid + 1},{r}) and ({l},{mid}) => P={fmt_p(P)}, s={s}")
    return s

# ---- the other sums of the course code, with the depth ----
depth = 0
max_depth = 0

def sum_it(a, n):
    s = 0
    for i in range(n):
        s = s + a[i]
    return s

def sum_rec_halving(a, l, r):
    global depth, max_depth
    depth += 1
    max_depth = max(max_depth, depth)
    if l == r:
        result = a[l]
    else:
        mid = l + (r - l) // 2
        result = sum_rec_halving(a, l, mid) + sum_rec_halving(a, mid + 1, r)
    depth -= 1
    return result

def sum_rec_decrease(a, n):
    global depth, max_depth
    depth += 1
    max_depth = max(max_depth, depth)
    if n == 0:
        result = 0
    else:
        result = sum_rec_decrease(a, n - 1) + a[n - 1]
    depth -= 1
    return result

a = [4, 7, 1, 9]
n = len(a)
print("a =", a, "  n =", n)
print("SumExplicitStack(a, 4):")
res = sum_explicit_stack_trace(a, n)
max_p = 0
sum_explicit_stack(a, n)
print(f"result {res}, maximum size of P = {max_p}")
print()
print("sumIt                =", sum_it(a, n))
max_depth = 0
print("sumRecHalving        =", sum_rec_halving(a, 0, n - 1), "  maximum depth =", max_depth)
max_depth = 0
print("sumRecDecrease       =", sum_rec_decrease(a, n), "  maximum depth =", max_depth)

# ---- the call stack of sumRecDecrease(a, 3) ----
step = 0
max_active = 0

def emit(stack, action):
    global step, max_active
    step += 1
    max_active = max(max_active, len(stack))
    print(f"{step:>2d} | {' '.join(stack)} | {action}")

def sum_rec_decrease_trace(a, n, stack):
    stack.append(f"s({n})")
    if n == 0:
        emit(stack, "n=0: return 0")
        stack.pop()
        return 0
    emit(stack, f"n={n} != 0: call s({n - 1}), then add a[{n - 1}]")
    r = sum_rec_decrease_trace(a, n - 1, stack)
    emit(stack, f"return {r}+a[{n - 1}]={r + a[n - 1]}")
    stack.pop()
    return r + a[n - 1]

b = [4, 7, 1]
print()
print("a =", b, "  sumRecDecrease(a, 3)")
print(" # | stack of the active calls | action")
res = sum_rec_decrease_trace(b, 3, [])
print(f"result {res}, active calls at most = {max_active} = n+1")

a = [4, 7, 1, 9]   n = 4
SumExplicitStack(a, 4):
pop (0,3): mid=1, push (2,3) and (0,1) => P=[(2,3),(0,1)], s=0
pop (0,1): mid=0, push (1,1) and (0,0) => P=[(2,3),(1,1),(0,0)], s=0
pop (0,0): base case, s=4 => P=[(2,3),(1,1)]
pop (1,1): base case, s=11 => P=[(2,3)]
pop (2,3): mid=2, push (3,3) and (2,2) => P=[(3,3),(2,2)], s=11
pop (2,2): base case, s=12 => P=[(3,3)]
pop (3,3): base case, s=21 => P=[]
result 21, maximum size of P = 3

sumIt                = 21
sumRecHalving        = 21   maximum depth = 3
sumRecDecrease       = 21   maximum depth = 5

a = [4, 7, 1]   sumRecDecrease(a, 3)
 # | stack of the active calls | action
 1 | s(3) | n=3 != 0: call s(2), then add a[2]
 2 | s(3) s(2) | n=2 != 0: call s(1), then add a[1]
 3 | s(3) s(2) s(1) | n=1 != 0: call s(0), then add a[0]
 4 | s(3) s(2) s(1) s(0) | n=0: return 0
 5 | s(3) s(2) s(1) | return 0+a[0]=4
 6 | s(3) s(2) | return 4+a[1]=11
 7 | s(3) | return 11+a[2]=12
result 12, active calls at most = 4 = n+1


**Try it yourself.** Change the array: the explicit stack never contains more than $\lceil\log_2 n\rceil+1$ pairs.

In [2]:
# Change the array: the explicit stack never contains more than ceil(log2 n) + 1 pairs.
c = list(range(1, 101))
max_p = 0
print(sum_explicit_stack(c, len(c)), max_p)

5050 8


## 2. Recursive MergeSort and MergeSortBottomUp

**MergeSortBottomUp**$(a,n)$ — *Input*: an array $a[1..n]$ of natural numbers. *Output*: the array sorted in non-decreasing order. The same splitting as Merge Sort, but **without recursion**: first we merge pairs of sub-arrays of length $1$, then of length $2$, $4$, … with the procedure `Merge` of chapter 17 (repeated here). It executes $\lceil\log_2 n\rceil$ passes and $\Theta(n\log_2 n)$ steps, like the recursive version, and does not use the call stack.

First the example of the chapter $a=[5,2,4,6,1,3]$, pass by pass (indices are from 0). Then the comparison, on pseudo-random arrays (same generator in all languages, seed $12345$, values between $1$ and $10^6$), of the number of **comparisons** between elements of `InsertionSort`, `MergeSort` and `MergeSortBottomUp`: $\Theta(n^2)$ against $\Theta(n\log_2 n)$, with the two versions of Merge Sort practically equal ("OK" means that the three algorithms give the same sorted array), and the measured times.

In [3]:
import time

comparisons = 0   # comparisons between elements

def merge(a, l, m, r):
    """Merge(a, l, m, r) of chapter 17."""
    global comparisons
    n1 = m - l + 1
    n2 = r - m
    L = [0] * n1
    R = [0] * n2
    for i in range(n1):
        L[i] = a[l + i]
    for j in range(n2):
        R[j] = a[m + 1 + j]
    i = 0
    j = 0
    k = l
    while i < n1 and j < n2:
        comparisons += 1
        if L[i] <= R[j]:
            a[k] = L[i]
            i += 1
        else:
            a[k] = R[j]
            j += 1
        k += 1
    while i < n1:
        a[k] = L[i]
        i += 1
        k += 1
    while j < n2:
        a[k] = R[j]
        j += 1
        k += 1

def merge_sort(a, l, r):
    if l < r:
        m = l + (r - l) // 2
        merge_sort(a, l, m)
        merge_sort(a, m + 1, r)
        merge(a, l, m, r)

def merge_sort_bottom_up(a, n):
    """MergeSortBottomUp(a, n): sorts a[0..n-1] without recursion."""
    width = 1
    while width < n:
        l = 0
        while l + width < n:                    # pseudo-code: l + width <= n
            m = l + width - 1
            r = min(l + 2 * width - 1, n - 1)
            merge(a, l, m, r)
            l = l + 2 * width
        width = 2 * width

def merge_sort_bottom_up_trace(a, n):
    width = 1
    passes = 0
    while width < n:
        l = 0
        parts = []
        while l + width < n:
            m = l + width - 1
            r = min(l + 2 * width - 1, n - 1)
            merge(a, l, m, r)
            parts.append(f"a[{l}..{m}] with a[{m + 1}..{r}]")
            l = l + 2 * width
        passes += 1
        print(f"width={width}: merge " + ", ".join(parts) + f" => {a}")
        width = 2 * width
    print(f"passes = {passes}")

def insertion_sort(a, n):
    """InsertionSort(a, n) (chapter 3), with the count of comparisons."""
    global comparisons
    for j in range(1, n):
        key = a[j]
        i = j - 1
        while i >= 0:
            comparisons += 1
            if a[i] > key:
                a[i + 1] = a[i]
                i = i - 1
            else:
                break
        a[i + 1] = key

def random_array(n, seed=12345):
    """Linear congruential generator: same numbers in Python, C++ and Java."""
    x = seed
    a = []
    for _ in range(n):
        x = (x * 1103515245 + 12345) % 2147483648
        a.append(1 + x % 1000000)
    return a

a = [5, 2, 4, 6, 1, 3]
print("a =", a, "  n =", len(a))
merge_sort_bottom_up_trace(a, len(a))

print()
print(f"{'n':>6s} {'InsertionSort':>14s} {'MergeSort':>10s} {'BottomUp':>9s}  comparisons")
times = []
for n in [1000, 2000, 4000]:
    base = random_array(n)
    x1, x2, x3 = base[:], base[:], base[:]
    comparisons = 0
    t0 = time.perf_counter(); insertion_sort(x1, n); t1 = (time.perf_counter() - t0) * 1000
    c1 = comparisons
    comparisons = 0
    t0 = time.perf_counter(); merge_sort(x2, 0, n - 1); t2 = (time.perf_counter() - t0) * 1000
    c2 = comparisons
    comparisons = 0
    t0 = time.perf_counter(); merge_sort_bottom_up(x3, n); t3 = (time.perf_counter() - t0) * 1000
    c3 = comparisons
    ok = "OK" if (x1 == x2 == x3 == sorted(base)) else "ERROR"
    print(f"{n:>6d} {c1:>14d} {c2:>10d} {c3:>9d}  {ok}")
    times.append((n, t1, t2, t3))
print()
print("Measured times (they depend on the machine):")
for n, t1, t2, t3 in times:
    print(f"n={n:<6d} InsertionSort {t1:>10.3f} ms   MergeSort {t2:>8.3f} ms   MergeSortBottomUp {t3:>8.3f} ms")

a = [5, 2, 4, 6, 1, 3]   n = 6
width=1: merge a[0..0] with a[1..1], a[2..2] with a[3..3], a[4..4] with a[5..5] => [2, 5, 4, 6, 1, 3]
width=2: merge a[0..1] with a[2..3] => [2, 4, 5, 6, 1, 3]
width=4: merge a[0..3] with a[4..5] => [1, 2, 3, 4, 5, 6]
passes = 3

     n  InsertionSort  MergeSort  BottomUp  comparisons
  1000         248782       8716      8740  OK
  2000        1010300      19415     19438  OK
  4000        4027558      42874     42876  OK

Measured times (they depend on the machine):
n=1000   InsertionSort     14.113 ms   MergeSort    1.161 ms   MergeSortBottomUp    1.120 ms
n=2000   InsertionSort     47.253 ms   MergeSort    2.090 ms   MergeSortBottomUp    2.059 ms
n=4000   InsertionSort    162.482 ms   MergeSort    4.128 ms   MergeSortBottomUp    4.126 ms


**Try it yourself.** Change $n$ (with $n=8000$ the comparisons of `InsertionSort` quadruple with respect to $n=4000$, those of Merge Sort little more than double).

In [4]:
# Try n = 8000: the comparisons of InsertionSort quadruple with respect to n = 4000, those of Merge Sort little more than double.
n = 8000
base = random_array(n)
x1, x2 = base[:], base[:]
comparisons = 0
insertion_sort(x1, n)
c1 = comparisons
comparisons = 0
merge_sort_bottom_up(x2, n)
print(c1, comparisons, x1 == x2)

16115652 93756 True


## 3. Fast power: recursive and PowerIt (square-and-multiply)

- **FastPower**$(x,n)$ (capitolo 14) — recursive: $x^n=(x^{\lfloor n/2\rfloor})^2$ if $n$ is even, $(x^{\lfloor n/2\rfloor})^2\cdot x$ if it is odd. $\Theta(\log_2 n)$ steps, $\lfloor\log_2 n\rfloor+2$ calls (depth $\Theta(\log_2 n)$). The result of the recursive call is stored in $y$ and used twice.
- **PowerIt**$(x,n)$ — *Input*: a number $x$ and a natural number $n$. *Output*: $x^n$. It reads the bits of $n$ from the least significant one: $\lfloor\log_2 n\rfloor+1$ iterations, extra memory $\Theta(1)$; $p\cdot y^k=x^n$ holds at every iteration.

With $x=2$ and $n=13=(1101)_2$ we print the trace of the chapter: $p$ is multiplied by $x$, $x^4$ and $x^8$ (the bits of 13 equal to 1) and $p=x^{13}$ after $4$ iterations. Then the table: same result, $\lfloor\log_2 n\rfloor+1$ iterations against $\lfloor\log_2 n\rfloor+2$ calls, against the $n$ multiplications of the naive loop.

In [5]:
pow_calls = 0   # calls of FastPower
pow_iters = 0   # iterations of PowerIt

def fast_power(x, n):
    """FastPower(x, n): x^n by recursion on the halving of n."""
    global pow_calls
    pow_calls += 1
    if n == 0:
        return 1                    # base case
    y = fast_power(x, n // 2)       # one recursive call
    if n % 2 == 0:
        return y * y                # n even
    return x * y * y                # n odd

def power_it(x, n):
    """PowerIt(x, n): x^n with square-and-multiply."""
    global pow_iters
    p = 1
    y = x
    k = n
    while k > 0:
        pow_iters += 1
        if k % 2 == 1:
            p = p * y
        y = y * y
        k = k // 2
    return p

def power_it_trace(x, n):
    print(f"PowerIt({x}, {n}): {n} = ({bin(n)[2:]})_2")
    p = 1
    y = x
    k = n
    it = 0
    while k > 0:
        it += 1
        par = "odd" if k % 2 == 1 else "even"
        if k % 2 == 1:
            p = p * y
        y = y * y
        print(f"{it}. k={k} {par}: p={p}, y={y}, k={k // 2}")
        k = k // 2
    print(f"iterations = {it}, result p = {p}")

power_it_trace(2, 13)
print()
print(f"{'n':>3s} {'2^n':>12s} {'PowerIt':>8s} {'FastPower':>10s} {'naive':>6s}   (iterations, calls, multiplications)")
for n in [0, 1, 2, 5, 13, 30]:
    pow_iters = 0
    r1 = power_it(2, n)
    pow_calls = 0
    r2 = fast_power(2, n)
    print(f"{n:>3d} {r1:>12d} {pow_iters:>8d} {pow_calls:>10d} {n:>6d}   {'OK' if r1 == r2 == 2 ** n else 'ERROR'}")

PowerIt(2, 13): 13 = (1101)_2
1. k=13 odd: p=2, y=4, k=6
2. k=6 even: p=2, y=16, k=3
3. k=3 odd: p=32, y=256, k=1
4. k=1 odd: p=8192, y=65536, k=0
iterations = 4, result p = 8192

  n          2^n  PowerIt  FastPower  naive   (iterations, calls, multiplications)
  0            1        0          1      0   OK
  1            2        1          2      1   OK
  2            4        2          3      2   OK
  5           32        3          4      5   OK
 13         8192        4          5     13   OK
 30   1073741824        5          6     30   OK


**Try it yourself.** Try another base and another exponent (in C++ and Java the result must fit in 64 bits).

In [6]:
# Try another base and another exponent (beware: in C++ and Java the result must fit in 64 bits).
pow_iters = 0
pow_calls = 0
print(power_it(3, 25), pow_iters, fast_power(3, 25), pow_calls)

847288609443 5 847288609443 6


## 4. Kadane: a better iterative algorithm

**Kadane**$(a,n)$ — *Input*: an array $a[1..n]$ of $n\ge1$ integers. *Output*: the maximum sum of a non-empty subarray $a[l..r]$. It is the compact version of the chapter: $cur\leftarrow\max(a[i],\,cur+a[i])$, $best\leftarrow\max(best,cur)$. The loop makes $n-1$ iterations: $\Theta(n)$ steps and $\Theta(1)$ memory, against the $n(n+1)/2$ pairs $(l,r)$ of brute force ($\Theta(n^2)$) and against the recursive divide and conquer ($\Theta(n\log_2 n)$): a good iterative algorithm can beat recursion.

On the example of the price changes of chapter 17 the result is $43$. Then, on pseudo-random arrays with values between $-50$ and $50$ (same generator in all languages, seed $n$), the iterations of `Kadane`, the pairs examined by brute force and the check that the two algorithms find the same maximum sum; finally the measured times.

In [7]:
import time

def kadane(a, n):
    """Kadane(a, n): maximum sum of a non-empty subarray of a[0..n-1]."""
    best = a[0]
    cur = a[0]                      # maximum sum of a subarray ending at i
    i = 1
    while i < n:
        cur = max(a[i], cur + a[i])
        best = max(best, cur)
        i = i + 1
    return best

def brute_force_best(a, n):
    """Brute force: maximum sum and number of pairs (l, r) examined."""
    best = a[0]
    pairs = 0
    for l in range(n):
        s = 0
        for r in range(l, n):
            s = s + a[r]
            pairs += 1
            if s > best:
                best = s
    return best, pairs

def random_values(n, seed=12345):
    """Integers between -50 and 50 (linear congruential generator)."""
    x = seed
    a = []
    for _ in range(n):
        x = (x * 1103515245 + 12345) % 2147483648
        a.append(x % 101 - 50)
    return a

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
print("a =", a)
print("Kadane(a, 16) =", kadane(a, len(a)))
b = [-3, -1, -2, -5]
print("a =", b, " Kadane(a, 4) =", kadane(b, len(b)))
print()
print(f"{'n':>6s} {'Kadane':>7s} {'BruteForce':>11s} {'iterations':>11s} {'pairs':>9s}  same sum?")
times = []
for n in [1000, 2000, 4000]:
    v = random_values(n, n)
    t0 = time.perf_counter(); k = kadane(v, n); t1 = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter(); bf, pairs = brute_force_best(v, n); t2 = (time.perf_counter() - t0) * 1000
    print(f"{n:>6d} {k:>7d} {bf:>11d} {n - 1:>11d} {pairs:>9d}  {'OK' if k == bf else 'ERROR'}")
    times.append((n, t1, t2))
print()
print("Measured times (they depend on the machine):")
for n, t1, t2 in times:
    print(f"n={n:<6d} Kadane {t1:>9.4f} ms   BruteForce {t2:>10.3f} ms")

a = [13, -3, -25, 20, -3, -16, -23, 18, 20, -7, 12, -5, -22, 15, -4, 7]
Kadane(a, 16) = 43
a = [-3, -1, -2, -5]  Kadane(a, 4) = -1

     n  Kadane  BruteForce  iterations     pairs  same sum?
  1000     547         547         999    500500  OK
  2000     981         981        1999   2001000  OK
  4000    1955        1955        3999   8002000  OK

Measured times (they depend on the machine):
n=1000   Kadane    0.0585 ms   BruteForce     14.648 ms
n=2000   Kadane    0.1155 ms   BruteForce     59.542 ms
n=4000   Kadane    0.2243 ms   BruteForce    249.844 ms


**Try it yourself.** Change $n$: the pairs of brute force grow as $n^2$, the iterations of Kadane as $n$.

In [8]:
# Try n = 8000: the pairs of brute force are n(n+1)/2 = 32 004 000, the iterations of Kadane 7999.
n = 8000
v = random_values(n)
print(kadane(v, n), brute_force_best(v, n))

1903 (1903, 32004000)


## 5. Fibonacci: FibRec, FibMemo and FibIt (overlapping subproblems)

*Input*: a natural number $n$. *Output*: the Fibonacci number $F(n)$, with $F(0)=0$, $F(1)=1$, $F(n)=F(n-1)+F(n-2)$.

- **FibRec**$(n)$ — direct translation of the definition: the subproblems overlap and the calls are $2F(n+1)-1=\Theta(\varphi^n)$, exponential; the depth is $n$.
- **FibMemo**$(n,memo)$ — recursion with memoization: every value is computed once (the vector `memo` is $-1$ for "undefined"); $2n-1$ calls, memory $\Theta(n)$.
- **FibIt**$(n)$ — loop with two variables: $n-1$ iterations, memory $\Theta(1)$.

On the example $\texttt{FibRec}(5)$ of the chapter we count how many times each subproblem is called (the tree has $15$ nodes, but only $6$ different values are needed). Then the table of the counts (which coincide with the values of the chapter: $21\,891$, $242\,785$, $2\,692\,537$ calls for $n=20,25,30$) and the measured times.

In [9]:
import time

fib_calls = 0   # calls executed

def fib_rec(n):
    """FibRec(n)"""
    global fib_calls
    fib_calls += 1
    if n <= 1:
        return n
    return fib_rec(n - 1) + fib_rec(n - 2)

def fib_memo(n, memo):
    """FibMemo(n, memo): memo[k] = -1 means "undefined"."""
    global fib_calls
    fib_calls += 1
    if n <= 1:
        return n
    if memo[n] == -1:
        memo[n] = fib_memo(n - 1, memo) + fib_memo(n - 2, memo)
    return memo[n]

def fib_it(n):
    """FibIt(n)"""
    if n <= 1:
        return n
    prev = 0                        # F(i-2)
    curr = 1                        # F(i-1)
    i = 2
    while i <= n:
        nxt = prev + curr
        prev = curr
        curr = nxt
        i = i + 1
    return curr

def fib_rec_hist(n, counts):
    """Like FibRec, but counts the calls for each value of the parameter."""
    counts[n] += 1
    if n <= 1:
        return n
    return fib_rec_hist(n - 1, counts) + fib_rec_hist(n - 2, counts)

counts = [0] * 6
print("FibRec(5) =", fib_rec_hist(5, counts))
print("calls for each value of the parameter:", "  ".join(f"{k}:{counts[k]}" for k in range(5, -1, -1)), f"  total {sum(counts)}")
print()
print(f"{'n':>3s} {'F(n)':>8s} {'FibRec':>9s} {'2F(n+1)-1':>10s} {'FibMemo':>8s} {'FibIt':>6s}  (calls, calls, calls, iterations)")
for n in [10, 15, 20, 25, 30]:
    f = fib_it(n)
    fib_calls = 0
    r1 = fib_rec(n)
    c1 = fib_calls
    fib_calls = 0
    r2 = fib_memo(n, [-1] * (n + 1))
    c2 = fib_calls
    ok = "OK" if (r1 == r2 == f and c1 == 2 * fib_it(n + 1) - 1 and c2 == 2 * n - 1) else "ERROR"
    print(f"{n:>3d} {f:>8d} {c1:>9d} {2 * fib_it(n + 1) - 1:>10d} {c2:>8d} {n - 1:>6d}  {ok}")
print()
print("Measured times (they depend on the machine):")
for n in [20, 25, 30]:
    t0 = time.perf_counter(); fib_rec(n); t1 = (time.perf_counter() - t0) * 1000
    t0 = time.perf_counter()
    for _ in range(1000):
        fib_it(n)
    t2 = (time.perf_counter() - t0) * 1000 / 1000
    print(f"n={n:<3d} FibRec {t1:>10.4f} ms   FibIt {t2:>10.6f} ms")

FibRec(5) = 5
calls for each value of the parameter: 5:1  4:1  3:2  2:3  1:5  0:3   total 15

  n     F(n)    FibRec  2F(n+1)-1  FibMemo  FibIt  (calls, calls, calls, iterations)
 10       55       177        177       19      9  OK
 15      610      1973       1973       29     14  OK
 20     6765     21891      21891       39     19  OK
 25    75025    242785     242785       49     24  OK
 30   832040   2692537    2692537       59     29  OK

Measured times (they depend on the machine):
n=20  FibRec     0.7783 ms   FibIt   0.000359 ms
n=25  FibRec     8.1920 ms   FibIt   0.000502 ms
n=30  FibRec    89.6841 ms   FibIt   0.000581 ms


**Try it yourself.** Increase $n$ (with $n=32$ `FibRec` makes $7\,049\,155$ calls, `FibMemo` $63$) and see how the time of `FibRec` grows.

In [10]:
# Increase n: with n = 32 FibRec makes 7 049 155 calls, FibMemo 63.
n = 32
fib_calls = 0
print(fib_rec(n), fib_calls)
fib_calls = 0
print(fib_memo(n, [-1] * (n + 1)), fib_calls)

2178309 7049155
2178309 63


## 6. Tail recursion and its elimination: GCDRec/GCDIt, RecBinarySearch/ItBinarySearch

A recursive call is a *tail call* if it is the last operation and its result is returned as it is: it can be replaced by an update of the parameters and a jump to the beginning, that is, by a loop (loop condition = negation of the base case; tail call = simultaneous assignment of the parameters). The result is the same, the number of steps has the same order of growth and the extra memory goes down from $\Theta(\text{depth})$ to $\Theta(1)$.

- **GCDRec**$(a,b)$ / **GCDIt**$(a,b)$ — *Input*: two natural numbers $a$, $b$. *Output*: their greatest common divisor (Euclid's algorithm). With $(a,b)=(252,105)$ both go through the pairs $(252,105)$, $(105,42)$, $(42,21)$, $(21,0)$ and return $21$: four calls against three iterations. The temporary variable $r$ is needed because $a\bmod b$ must be computed with the *old* $a$.
- **RecBS**$(a,l,r,m)$ / **ItBS**$(a,l,r,m)$ — *Input*: a sorted array, two indices $l$, $r$ and a value $m$. *Output*: an index of $m$ in $a[l..r]$ or $-1$. `ItBS(a, 1, n, m)` is the iterative binary search of the chapter "Search".

In [11]:
gcd_log = []    # pairs (a, b) visited (only to print them)
rbs_calls = 0   # calls of RecBS
bs_iters = 0    # iterations of ItBS

def gcd_rec(a, b):
    """GCDRec(a, b)"""
    gcd_log.append(f"({a},{b})")
    if b == 0:
        return a
    return gcd_rec(b, a % b)            # tail call

def gcd_it(a, b):
    """GCDIt(a, b)"""
    gcd_log.append(f"({a},{b})")
    while b != 0:                       # negation of the base case
        r = a % b                       # with the old a
        a = b
        b = r
        gcd_log.append(f"({a},{b})")
    return a

def rec_bs(a, l, r, m):
    """RecBS(a, l, r, m)"""
    global rbs_calls
    rbs_calls += 1
    if l > r:
        return -1
    mid = l + (r - l) // 2
    if a[mid] == m:
        return mid
    if a[mid] < m:
        return rec_bs(a, mid + 1, r, m)     # tail call
    else:
        return rec_bs(a, l, mid - 1, m)     # tail call

def it_bs(a, l, r, m):
    """ItBS(a, l, r, m)"""
    global bs_iters
    while l <= r:                           # negation of the base case l > r
        bs_iters += 1
        mid = l + (r - l) // 2
        if a[mid] == m:
            return mid                      # the second base case stays in the loop
        if a[mid] < m:
            l = mid + 1
        else:
            r = mid - 1
    return -1

for a, b in [(252, 105), (1071, 462), (832040, 514229)]:
    gcd_log.clear()
    g1 = gcd_rec(a, b)
    calls = len(gcd_log)
    pairs = " ".join(gcd_log)
    gcd_log.clear()
    g2 = gcd_it(a, b)
    iters = len(gcd_log) - 1
    print(f"GCDRec({a}, {b}) = {g1}   calls={calls}     GCDIt = {g2}   iterations={iters}   {'OK' if g1 == g2 else 'ERROR'}")
    if a == 252:
        print("  pairs:", pairs)

print()
a = [3, 9, 11, 12, 12, 16, 19]
n = len(a)
print("a =", a, "  n =", n)
print(f"{'m':>4s} {'RecBS':>6s} {'calls':>6s} {'ItBS':>6s} {'iterations':>11s}")
for m in [12, 10, 3, 19, 100]:
    rbs_calls = 0
    r1 = rec_bs(a, 0, n - 1, m)
    bs_iters = 0
    r2 = it_bs(a, 0, n - 1, m)
    print(f"{m:>4d} {r1:>6d} {rbs_calls:>6d} {r2:>6d} {bs_iters:>11d}   {'OK' if r1 == r2 else 'ERROR'}")

GCDRec(252, 105) = 21   calls=4     GCDIt = 21   iterations=3   OK
  pairs: (252,105) (105,42) (42,21) (21,0)
GCDRec(1071, 462) = 21   calls=4     GCDIt = 21   iterations=3   OK
GCDRec(832040, 514229) = 1   calls=29     GCDIt = 1   iterations=28   OK

a = [3, 9, 11, 12, 12, 16, 19]   n = 7
   m  RecBS  calls   ItBS  iterations
  12      3      1      3           1   OK
  10     -1      4     -1           3   OK
   3      0      3      0           3   OK
  19      6      3      6           3   OK
 100     -1      4     -1           3   OK


**Try it yourself.** Try other pairs for the GCD: with two consecutive Fibonacci numbers Euclid's algorithm makes the largest number of steps.

In [12]:
# Try other pairs for the GCD: with two consecutive Fibonacci numbers Euclid's algorithm makes the largest number of steps.
gcd_log.clear()
print(gcd_rec(46368, 28657), len(gcd_log))

1 23


## 7. Depth of the recursion and the call stack

The memory used by the call stack is proportional to the **depth** $D(n)$ of the recursion (the maximum number of calls active at the same time), not to the number of calls. The table compares the **measured** depth with the formula of the chapter:

| algorithm | $D(n)$ |
|---|---|
| sum by decrease by one (base case $n=0$) | $n+1$ |
| sum by halving | $\lceil\log_2 n\rceil+1$ |
| FastPower | $\lfloor\log_2 n\rfloor+2$ |
| RecBS (worst case, absent value) | $\le\lfloor\log_2 n\rfloor+2$ |
| FibRec$(n)$, $n\ge1$ | $n$ |

A depth $\Theta(\log_2 n)$ is never a problem in practice; a depth $\Theta(n)$ risks a **stack overflow**: the last part of the cell (the lines starting with `~~`) depends on the language and on the machine.

In [13]:
import sys

depth = 0
max_depth = 0
calls = 0

def enter():
    global depth, max_depth
    depth += 1
    max_depth = max(max_depth, depth)

def sum_rec_decrease(a, n):
    enter()
    result = 0 if n == 0 else sum_rec_decrease(a, n - 1) + a[n - 1]
    exit_call()
    return result

def sum_rec_halving(a, l, r):
    enter()
    if l == r:
        result = a[l]
    else:
        mid = l + (r - l) // 2
        result = sum_rec_halving(a, l, mid) + sum_rec_halving(a, mid + 1, r)
    exit_call()
    return result

def fast_power(x, n):
    enter()
    if n == 0:
        result = 1
    else:
        y = fast_power(x, n // 2)
        result = y * y if n % 2 == 0 else x * y * y
    exit_call()
    return result

def rec_bs(a, l, r, m):
    global calls
    calls += 1
    if l > r:
        return -1
    mid = l + (r - l) // 2
    if a[mid] == m:
        return mid
    if a[mid] < m:
        return rec_bs(a, mid + 1, r, m)
    return rec_bs(a, l, mid - 1, m)

def fib_rec(n):
    enter()
    result = n if n <= 1 else fib_rec(n - 1) + fib_rec(n - 2)
    exit_call()
    return result

def exit_call():
    global depth
    depth -= 1

def floor_log2(n):
    return n.bit_length() - 1

def ceil_log2(n):
    return (n - 1).bit_length()

print(f"{'n':>5s} {'decrease':>9s} {'n+1':>5s} {'halving':>8s} {'formula':>8s} {'FastPower':>10s} {'formula':>8s} {'RecBS':>6s} {'bound':>6s}")
for n in [10, 100, 500]:
    a = list(range(n))
    max_depth = 0; sum_rec_decrease(a, n); d1 = max_depth
    max_depth = 0; sum_rec_halving(a, 0, n - 1); d2 = max_depth
    max_depth = 0; fast_power(1, n); d3 = max_depth
    calls = 0; rec_bs(a, 0, n - 1, n + 100); d4 = calls
    print(f"{n:>5d} {d1:>9d} {n + 1:>5d} {d2:>8d} {ceil_log2(n) + 1:>8d} {d3:>10d} {floor_log2(n) + 2:>8d} {d4:>6d} {floor_log2(n) + 2:>6d}")
print()
print(f"{'n':>3s} {'FibRec depth':>13s}")
for n in [5, 10, 20]:
    max_depth = 0; fib_rec(n)
    print(f"{n:>3d} {max_depth:>13d}")

print(f"~~ Python: recursion limit = {sys.getrecursionlimit()}")
try:
    depth = 0
    sum_rec_decrease([0] * 100000, 100000)
    print("~~ Python: n = 100000 OK")
except RecursionError:
    depth = 0
    print("~~ Python: RecursionError for sum_rec_decrease with n = 100000 (depth n+1 > limit)")

    n  decrease   n+1  halving  formula  FastPower  formula  RecBS  bound
   10        11    11        5        5          5        5      5      5
  100       101   101        8        8          8        8      8      8
  500       501   501       10       10         10       10     10     10

  n  FibRec depth
  5             5
 10            10
 20            20
~~ Python: recursion limit = 1000
~~ Python: RecursionError for sum_rec_decrease with n = 100000 (depth n+1 > limit)


**Try it yourself.** Measure the depth for another $n$ (here $2000$): `decrease` $=n+1$, `halving` $=\lceil\log_2 n\rceil+1$.

In [14]:
# Measure the depth of the recursion for another n (here 2000): decrease = n+1, halving = ceil(log2 n) + 1.
sys.setrecursionlimit(5000)         # a depth of 2001 calls is needed
n = 2000
a = [1] * n
depth = 0
max_depth = 0
sum_rec_decrease(a, n)
print(max_depth, end=" ")
max_depth = 0
sum_rec_halving(a, 0, n - 1)
print(max_depth)
sys.setrecursionlimit(1000)

2001 12
